# 08 - Camera Plate Polygon Labeling

Label plat dari foto kamera lokal untuk YOLO segmentation. Klik empat sudut plat mengikuti keliling objek, lalu pilih **Save & next**. File gambar asli tidak diubah.

## Cara penggunaan

1. Gunakan kernel `Python (.venv) - Smart Parking System`.
2. Jalankan seluruh cell sampai UI muncul.
3. Klik empat sudut plat secara berurutan mengelilingi plat, misalnya kiri atas, kanan atas, kanan bawah, kiri bawah.
4. Tekan **Save & next**.
5. Gunakan **Undo point** jika salah klik atau **Skip** jika gambar tidak layak dilabel.

Notebook otomatis melewati gambar yang sudah memiliki label sehingga pekerjaan dapat dilanjutkan kapan saja.

In [ ]:
%matplotlib widget

from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'ml').is_dir())
IMAGE_DIR = PROJECT_ROOT / 'data/raw/camera_plate_v1/images/unlabeled'
LABEL_DIR = PROJECT_ROOT / 'data/raw/camera_plate_v1/labels'
SKIP_FILE = PROJECT_ROOT / 'data/raw/camera_plate_v1/skipped.txt'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
LABEL_DIR.mkdir(parents=True, exist_ok=True)

image_paths = sorted(path for path in IMAGE_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
skipped = set(SKIP_FILE.read_text(encoding='utf-8').splitlines()) if SKIP_FILE.is_file() else set()
pending_paths = [path for path in image_paths if not (LABEL_DIR / f'{path.stem}.txt').is_file() and path.name not in skipped]
print(f'Total: {len(image_paths)} | Labeled: {len(image_paths) - len(pending_paths) - len(skipped)} | Skipped: {len(skipped)} | Pending: {len(pending_paths)}')

In [ ]:
class PolygonLabeler:
    def __init__(self, paths: list[Path]) -> None:
        self.paths = paths
        self.position = 0
        self.points: list[tuple[float, float]] = []
        self.image = None
        self.figure, self.axis = plt.subplots(figsize=(11, 7))
        self.figure.canvas.mpl_connect('button_press_event', self.on_click)
        self.status = widgets.HTML()
        self.save_button = widgets.Button(description='Save & next', button_style='success', disabled=True)
        self.undo_button = widgets.Button(description='Undo point', disabled=True)
        self.skip_button = widgets.Button(description='Skip')
        self.save_button.on_click(self.save_and_next)
        self.undo_button.on_click(self.undo_point)
        self.skip_button.on_click(self.skip_image)
        display(self.status, widgets.HBox([self.save_button, self.undo_button, self.skip_button]))
        self.show_current()

    def show_current(self) -> None:
        self.axis.clear()
        self.points = []
        if self.position >= len(self.paths):
            self.status.value = '<b>All pending images are complete.</b>'
            self.axis.set_axis_off()
            self.figure.canvas.draw_idle()
            self.save_button.disabled = True
            self.undo_button.disabled = True
            self.skip_button.disabled = True
            return
        path = self.paths[self.position]
        self.image = cv2.imread(str(path))
        if self.image is None:
            raise ValueError(f'Cannot read image: {path}')
        self.axis.imshow(cv2.cvtColor(self.image, cv2.COLOR_BGR2RGB))
        self.axis.set_title(path.name)
        self.axis.set_axis_off()
        self.update_status()
        self.figure.canvas.draw_idle()

    def update_status(self) -> None:
        completed = sum((LABEL_DIR / f'{path.stem}.txt').is_file() for path in image_paths)
        self.status.value = f'<b>Image {self.position + 1}/{len(self.paths)}</b> · points {len(self.points)}/4 · total labeled {completed}/{len(image_paths)}'
        self.save_button.disabled = len(self.points) != 4
        self.undo_button.disabled = not self.points

    def redraw_polygon(self) -> None:
        for line in list(self.axis.lines):
            line.remove()
        if self.points:
            values = np.asarray(self.points)
            self.axis.plot(values[:, 0], values[:, 1], 'o-', color='#f2d13d', linewidth=2, markersize=7)
            if len(self.points) == 4:
                closed = np.vstack([values, values[0]])
                self.axis.plot(closed[:, 0], closed[:, 1], '-', color='#f2d13d', linewidth=2)
        self.figure.canvas.draw_idle()
        self.update_status()

    def on_click(self, event) -> None:
        if event.inaxes != self.axis or event.xdata is None or len(self.points) >= 4:
            return
        self.points.append((float(event.xdata), float(event.ydata)))
        self.redraw_polygon()

    def undo_point(self, _button=None) -> None:
        if self.points:
            self.points.pop()
            self.redraw_polygon()

    def save_and_next(self, _button=None) -> None:
        if len(self.points) != 4:
            return
        height, width = self.image.shape[:2]
        normalized = [(min(max(x / width, 0.0), 1.0), min(max(y / height, 0.0), 1.0)) for x, y in self.points]
        coordinates = ' '.join(f'{value:.6f}' for point in normalized for value in point)
        path = self.paths[self.position]
        (LABEL_DIR / f'{path.stem}.txt').write_text(f'0 {coordinates}\n', encoding='utf-8')
        self.position += 1
        self.show_current()

    def skip_image(self, _button=None) -> None:
        path = self.paths[self.position]
        skipped.add(path.name)
        SKIP_FILE.write_text('\n'.join(sorted(skipped)) + '\n', encoding='utf-8')
        self.position += 1
        self.show_current()

labeler = PolygonLabeler(pending_paths)

## Audit progress

Jalankan ulang cell ini kapan saja untuk mengecek jumlah label dan validitas koordinat.

In [ ]:
label_paths = sorted(LABEL_DIR.glob('*.txt'))
invalid = []
for label_path in label_paths:
    parts = label_path.read_text(encoding='utf-8').split()
    try:
        values = [float(value) for value in parts[1:]]
        valid = parts[0] == '0' and len(values) == 8 and all(0.0 <= value <= 1.0 for value in values)
    except (ValueError, IndexError):
        valid = False
    if not valid:
        invalid.append(label_path.name)
print(f'Images: {len(image_paths)} | Labels: {len(label_paths)} | Skipped: {len(skipped)} | Invalid: {len(invalid)}')
invalid[:10]